# Relation Extraction & Knowledge Graph Construction

NER found entities. ENtity linking anchored them. Relation extraction finds the edges between them. A knowledge graph is the sum of nodes, edges and their provenance.

## Problem definition

Relation Extraction turns free text into structured triples (subject, relation, object). Aggreagate across a corpus and you have a knowledge graph.

"Tim Cook became CEO of Apple in 2011"

||subject|relation|object|
|---|---|---|---|
||Tim Cook|role|CEO|
||Tim Cook|employer|Apple|
||Tim Cook|start_date|2011|
||Apple|type|Organization|

LLMs extract relations enthusiastically. Too enthusiastically. They hallucinate triples that the source text does not support.

## Basic Concept

### Triple form.

`subject_entity, relation_type, object_entity`

Relations come from either:
1. A close ontology. (Wididata properities...)
2. Open set (anything goes)

### Extraction

#### Rule/Pattern -base 

Hearst patterns: "X such as Y" --> (Y, isA, X)

Brittle, percise, explainable

#### Supervised classifier

Given two entity mentions in a sentence, predict the relation from a fixed set.

#### Generative LLM

Prompt the model to emit triples.

### AEVS
* Anchor.  Indentify every entity span and relation-phrase span with exact positions.
* Extract. Generate triples linked to anchor spans.
* Verify. Match each triple element back to the source text. Reject anything unsupported.
* Supplement.  A coverage pass ensures no anchored span is dropped.

# Build your Own

## Pattern-based Extraction

In [5]:
import sys
from pathlib import Path
import re

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import SectionPrinter

PATTERNS = [
    (r"(?P<s>[A-Z]\w+) (?:is|was) (?:a|an|the) (?P<o>[A-Z]?\w+)", "isA"),
    (r"(?P<s>[A-Z]\w+) (?:is|was) born in (?P<o>\w+)", "bornIn"),
    (r"(?P<s>[A-Z]\w+) works? (?:at|for) (?P<o>[A-Z]\w+)", "worksAt"),
    (r"(?P<s>[A-Z]\w+) founded (?P<o>[A-Z]\w+)", "founded"),
]

with SectionPrinter("Pattern-based Extraction"):
    input = "Apple is a fruit."
    for pattern, rel in PATTERNS:
        pattern = re.compile(pattern)
        for m in pattern.finditer(input):
            subj = m.group(1)
            obj = m.group(2)
            span = (m.start(), m.end())

            print({
                "subject": subj,
                "relation": rel,
                "object": obj,
                "span": span,
            })


==================Pattern-based Extraction==================
{'subject': 'Apple', 'relation': 'isA', 'object': 'fruit', 'span': (0, 16)}


## Supervised relation classfication

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tok = AutoTokenizer.from_pretrained("Babelscape/rebel-large")
model = AutoModelForSeq2SeqLM.from_pretrained("Babelscape/rebel-large")

text = "Tim Cook was born in Alabama. He later became CEO of Apple."

encoded = tok(text, return_tensors="pt", truncation=True)
output = model.generate(**encoded, max_length=200)
triples = tok.batch_decode(output, skip_special_tokens=True)

with SectionPrinter("Supervised relation classification"):
    print(triples)

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

=============Supervised relation classification=============
 Tim Cook  Apple  employer


## LLM-prompted extration

In [7]:
prompt = f"""Extract (subject, relation, object) triples from the text.
For each triple, include the exact character span in the source text.

Text: {text}

Output JSON:
[{{"subject": {{"text": "...", "span": [start, end]}},
   "relation": "...",
   "object": {{"text": "...", "span": [start, end]}}}}, ...]

Only include triples fully supported by the text. No inference beyond what is stated.
"""


from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

agent = create_agent(
    init_chat_model(
        "deepseek:deepseek-chat",
        extra_body={"thinking": {"type": "disabled"}},
    ),
)

response = agent.invoke({"messages": [{"role": "user", "content": prompt}]})

with SectionPrinter("Structured output APIs"):
    print(response["messages"][-1].content)
    
    

===================Structured output APIs===================
```json
[
  {
    "subject": {"text": "Tim Cook", "span": [0, 8]},
    "relation": "was born in",
    "object": {"text": "Alibama", "span": [17, 24]}
  },
  {
    "subject": {"text": "He", "span": [28, 30]},
    "relation": "became",
    "object": {"text": "CEO of Apple", "span": [40, 52]}
  }
]
```
